# Feature Exploration
**The Sentiment Signal**  
Kevin (yy6074), Kelly (lx2455), Yuwei (yz13019)  
Prepared 23 October 2026.

This notebook completes the Week 2 feature-exploration work using NumPy and Pandas.
It inspects the cleaned dataset, demonstrates manipulation and exports supporting tables.
The scope ends at tabular profiling and feature preparation. Charts, relationship
interpretation, hypothesis testing, final feature selection and model training are
left for subsequent work by the group.

Input: a byte-identical snapshot of `../Data Cleaning/sentiment_cleaned.csv`, retained
locally in `data/sentiment_cleaned.csv` so this folder can run independently. The source
has 710 simulated posts and 26 columns. The publisher discloses ChatGPT-generated
synthetic content. Supplied sentiment labels are retained without relabeling.

In [1]:
from pathlib import Path
import json, hashlib, re
import numpy as np
import pandas as pd
from IPython.display import display

BASE=Path.cwd()
if not (BASE/'data/sentiment_cleaned.csv').exists():
    BASE=BASE/'Feature Exploration'
INPUT=BASE/'data/sentiment_cleaned.csv'
EXPECTED_INPUT_SHA256='4c38dc7cc2a8a06a96c13e989541c024ce31e9ee6af890a5f429fbe1f892ec6a'
input_hash=hashlib.sha256(INPUT.read_bytes()).hexdigest()
assert input_hash==EXPECTED_INPUT_SHA256, 'Input changed. Review the snapshot and update the documented source before refreshing.'
source=pd.read_csv(INPUT)
df=source.copy(deep=True)
(BASE/'tables').mkdir(exist_ok=True)
def save_table(name,frame):
    frame.to_csv(BASE/'tables'/f'{name}.csv',index=False)
    return frame
print('Input shape:',source.shape)
print('Source SHA256:',input_hash)
display(source.head(3))

Input shape: (710, 26)
Source SHA256: 4c38dc7cc2a8a06a96c13e989541c024ce31e9ee6af890a5f429fbe1f892ec6a


,source_record,source_index_1,source_index_2,text,sentiment,timestamp,user,platform,hashtags,retweets,...,text_lower,text_group_id,repeated_text,text_char_count,text_word_count,total_interactions,year_month,weekday,hashtag_list,hashtag_count
0,2,0,0,Enjoying a beautiful day at the park!,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15,...,enjoying a beautiful day at the park!,4eb36f41f4830c73,False,37,7,45,2023-01,Sunday,"[""nature"", ""park""]",2
1,3,1,1,Traffic was terrible this morning.,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5,...,traffic was terrible this morning.,abf4a5e3c9bee869,False,34,5,15,2023-01,Sunday,"[""morning"", ""traffic""]",2
2,4,2,2,Just finished an amazing workout! 💪,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20,...,just finished an amazing workout! 💪,1358f4dea8c87238,False,35,6,60,2023-01,Sunday,"[""fitness"", ""workout""]",2


## 1 Schema inspection and type conversion
Inspect shape, column names, types, completeness and cardinality. Convert dates,
whole-number measurements, booleans and repeated categories to appropriate in-memory
types. CSV exports require type parsing again on reload. Export indices and row IDs
remain provenance, not measured features.

In [2]:
before_types=source.dtypes.astype(str)
df['timestamp']=pd.to_datetime(df.timestamp,format='%Y-%m-%d %H:%M:%S',errors='raise')
integer_fields=['source_record','source_index_1','source_index_2','likes','retweets','year','month','day','hour',
                'text_char_count','text_word_count','total_interactions','hashtag_count']
for col in integer_fields:
    values=pd.to_numeric(df[col],errors='raise')
    assert values.mod(1).eq(0).all()
    df[col]=values.astype('Int64')
bool_map={'true':True,'false':False}
parsed_bool=df.repeated_text.astype(str).str.lower().map(bool_map)
assert parsed_bool.notna().all()
df['repeated_text']=parsed_bool.astype('boolean')
for col in ['sentiment','platform','country','weekday']:
    df[col]=df[col].astype('category')
df['hashtag_list']=df.hashtag_list.map(json.loads)
assert df.hashtag_list.map(lambda x:isinstance(x,list) and all(isinstance(t,str) for t in x)).all()
inventory=pd.DataFrame({
    'feature':source.columns,'csv_inferred_dtype':before_types.values,
    'prepared_dtype':[str(df[c].dtype) for c in source],
    'missing_cells':source.isna().sum().values,
    'unique_values':[source[c].nunique(dropna=True) for c in source],
})
inventory['unique_share_pct']=inventory.unique_values/len(df)*100
save_table('feature_inventory',inventory)
display(inventory)

,feature,csv_inferred_dtype,prepared_dtype,missing_cells,unique_values,unique_share_pct
0,source_record,int64,Int64,0,710,100.000000
1,source_index_1,int64,Int64,0,710,100.000000
2,source_index_2,int64,Int64,0,710,100.000000
3,text,object,object,0,706,99.436620
4,sentiment,object,category,0,191,26.901408
5,timestamp,object,datetime64[ns],0,683,96.197183
6,user,object,object,0,670,94.366197
7,platform,object,category,0,3,0.422535
8,hashtags,object,object,0,692,97.464789
9,retweets,int64,Int64,0,26,3.661972


## 2 Indexing filtering sorting and derived columns
Demonstrate `.loc`, `.assign`, `.sort_values` and vectorized string operations.
Filters below are worked examples and do not remove rows from the analytical dataset.
A count threshold of 75 interactions is an explicit query example, not a calibrated
popularity definition. Hour periods use the source timestamp without timezone conversion.

In [3]:
query_example=df.loc[(df.platform=='Instagram') & (df.total_interactions>=75),
    ['source_record','platform','text','likes','retweets','total_interactions']]
query_example=query_example.sort_values(['total_interactions','source_record'],ascending=[False,True])
save_table('filter_sort_example',query_example)
display(query_example.head(8))
hours=df.hour.to_numpy(dtype=int)
df=df.assign(
    hour_period=np.select([hours<6,hours<12,hours<18],['00–05','06–11','12–17'],default='18–23'),
    is_weekend=np.isin(df.timestamp.dt.weekday.to_numpy(),[5,6]),
    log1p_interactions=np.log1p(df.total_interactions.to_numpy(dtype=float)),
    has_url=df.text.str.contains(r'https?://|www\.',case=False,regex=True),
    has_mention=df.text.str.contains(r'(?<!\w)@\w+',regex=True),
    negation_cue_count=df.text.str.count(r"(?i)\b(?:no|not|never|neither|nor|cannot)\b|n['’]t\b"),
)
derived_cols=['hour_period','is_weekend','log1p_interactions','has_url','has_mention','negation_cue_count']
print('Added exploratory columns:',derived_cols)
display(df[['source_record','hour']+derived_cols].head(8))

,source_record,platform,text,likes,retweets,total_interactions
317,337,Instagram,Thrilled to witness the grandeur of a cultural...,80,40,120
350,370,Instagram,Elation over discovering a rare book in a quai...,80,40,120
364,384,Instagram,A sense of wonder at the vastness of the cosmo...,80,40,120
384,404,Instagram,Awe-inspired by the vastness of the cosmos on ...,80,40,120
410,434,Instagram,"Heartache deepens, a solitary journey through ...",80,40,120
448,472,Instagram,"Dancing on sunshine, each step a celebration o...",80,40,120
459,483,Instagram,"Surrounded by the colors of joy, a canvas pain...",80,40,120
488,512,Instagram,"At the front row of Adele's concert, each note...",80,40,120


Added exploratory columns: ['hour_period', 'is_weekend', 'log1p_interactions', 'has_url', 'has_mention', 'negation_cue_count']


,source_record,hour,hour_period,is_weekend,log1p_interactions,has_url,has_mention,negation_cue_count
0,2,12,12–17,True,3.828641,False,False,0
1,3,8,06–11,True,2.772589,False,False,0
2,4,15,12–17,True,4.110874,False,False,0
3,5,18,18–23,True,3.178054,False,False,0
4,6,19,18–23,True,3.637586,False,False,0
5,7,9,06–11,False,4.330733,False,False,0
6,8,14,12–17,False,3.433987,False,False,0
7,9,19,18–23,False,3.828641,False,False,0


`has_url` and `has_mention` flag simple patterns in text. `negation_cue_count` counts
listed words and common n’t contractions. These are reproducible lexical diagnostics,
not language understanding or a sentiment score. Absence of a flag does not establish
that a text lacks a URL, reference or negation expressed another way.

## 3 Numeric feature profiles with NumPy and Pandas
Summarize observed measurement values, not record IDs or calendar codes. Standard
deviations below use ddof=1, the sample convention used by Pandas. Quantiles use
NumPy's linear interpolation. No scaling parameters or model vocabulary are fitted.

In [4]:
measured=['likes','retweets','total_interactions','text_char_count','text_word_count','hashtag_count']
summary=[]
for col in measured:
    x=df[col].to_numpy(dtype=float)
    q1,median,q3=np.quantile(x,[.25,.5,.75],method='linear')
    summary.append({'feature':col,'count':len(x),'minimum':np.min(x),'q1':q1,'median':median,'q3':q3,
                    'maximum':np.max(x),'mean':np.mean(x),'sample_std':np.std(x,ddof=1),
                    'zero_count':np.count_nonzero(x==0),'unique_values':np.unique(x).size})
numeric_profile=save_table('numeric_profile',pd.DataFrame(summary))
for row in summary:
    assert np.isclose(row['mean'],df[row['feature']].mean())
    assert np.isclose(row['sample_std'],df[row['feature']].std(ddof=1))
display(numeric_profile.round(3))
numeric_candidates=measured+['log1p_interactions','negation_cue_count']
constants=pd.DataFrame([{'feature':c,'constant_value':str(df[c].iloc[0]),'unique_values':df[c].nunique()}
                       for c in numeric_candidates+['has_url','has_mention','is_weekend','hour_period'] if df[c].nunique()==1])
save_table('constant_feature_flags',constants)
display(constants)

,feature,count,minimum,q1,median,q3,maximum,mean,sample_std,zero_count,unique_values
0,likes,710,10.0,35.0,43.0,50.0,80.0,42.982,14.203,0,38
1,retweets,710,5.0,18.0,22.0,25.0,40.0,21.549,7.118,0,26
2,total_interactions,710,15.0,53.0,65.0,75.0,120.0,64.531,21.313,0,45
3,text_char_count,710,27.0,56.0,81.5,104.0,156.0,83.692,31.220,0,124
4,text_word_count,710,4.0,9.0,13.0,17.0,25.0,13.086,4.851,0,22
5,hashtag_count,710,2.0,2.0,2.0,2.0,2.0,2.000,0.000,0,1


,feature,constant_value,unique_values
0,hashtag_count,2,1
1,has_url,False,1
2,has_mention,False,1


## 4 Category frequencies and simple grouped operations
Value counts describe support for supplied labels and context fields. Means and
medians in the platform aggregation demonstrate `groupby().agg()`; they are not
significance tests or evidence of real-world platform behavior.

In [5]:
counts={}
for col in ['sentiment','platform','country','weekday','hour_period']:
    freq=df[col].value_counts(dropna=False).rename_axis(col).reset_index(name='posts')
    freq['share_pct']=freq.posts/len(df)*100
    assert freq.posts.sum()==len(df)
    counts[col]=save_table(col+'_value_counts',freq)
category_summary=save_table('category_summary',pd.DataFrame([
    {'feature':c,'unique_values':df[c].nunique(),'most_frequent_value':str(counts[c].iloc[0,0]),
     'most_frequent_posts':int(counts[c].iloc[0].posts)} for c in counts]))
platform_aggregation=df.groupby('platform',observed=True).agg(
    posts=('source_record','size'),mean_likes=('likes','mean'),median_likes=('likes','median'),
    mean_retweets=('retweets','mean'),median_retweets=('retweets','median'),
    median_interactions=('total_interactions','median')).reset_index()
save_table('platform_aggregation_example',platform_aggregation)
assert platform_aggregation.posts.sum()==len(df)
assert np.isclose((platform_aggregation.mean_likes*platform_aggregation.posts).sum(),df.likes.sum())
cross=pd.crosstab(df.platform,df.country)
save_table('platform_country_crosstab',cross.reset_index())
assert cross.to_numpy().sum()==len(df)
display(category_summary)
display(platform_aggregation.round(2))
display(cross.iloc[:,:6])
print('Crosstab preview shows six location columns; the exported table includes all locations.')

,feature,unique_values,most_frequent_value,most_frequent_posts
0,sentiment,191,Positive,45
1,platform,3,Instagram,251
2,country,33,USA,184
3,weekday,7,Sunday,113
4,hour_period,4,12–17,318


,platform,posts,mean_likes,median_likes,mean_retweets,median_retweets,median_interactions
0,Facebook,223,41.86,45.0,20.96,22.0,67.0
1,Instagram,251,45.24,45.0,22.68,22.0,67.0
2,Twitter,236,41.64,40.0,20.9,20.0,60.0


country,Australia,Austria,Belgium,Brazil,Cambodia,Canada
platform,,,,,,
Facebook,23,1,0,6,0,43
Instagram,26,0,0,6,1,42
Twitter,21,0,2,5,0,46


Crosstab preview shows six location columns; the exported table includes all locations.


## 5 Date text hashtag and user exploration
These tables establish available values and row support for colleagues to visualize
later. Calendar counts measure dataset coverage. Text word counts are whitespace
token counts rather than linguistic word counts. Hash tags are counted once per
post, so hashtag post shares can sum above 100%.

In [6]:
coverage=df.groupby(df.timestamp.dt.year).agg(posts=('source_record','size'),
    first_timestamp=('timestamp','min'),last_timestamp=('timestamp','max')).reset_index(names='year')
save_table('annual_coverage',coverage)
user_frequency=df.groupby('user',observed=True).size().rename('posts').reset_index().sort_values(['posts','user'],ascending=[False,True])
save_table('user_post_counts',user_frequency)
repeated=df.loc[df.repeated_text,['source_record','text_group_id','text','sentiment','platform','timestamp']]
save_table('repeated_text_records',repeated)
tags=df[['source_record','hashtag_list']].explode('hashtag_list').rename(columns={'hashtag_list':'hashtag'})
assert tags.hashtag.notna().all()
assert not tags.duplicated(['source_record','hashtag']).any()
tags=tags.merge(df[['source_record','platform']],on='source_record',how='left',validate='many_to_one')
save_table('post_hashtags_long',tags)
tag_frequency=tags.hashtag.value_counts().rename_axis('hashtag').reset_index(name='posts')
tag_frequency['post_share_pct']=tag_frequency.posts/len(df)*100
save_table('hashtag_post_counts',tag_frequency)
support_bands=pd.cut(counts['sentiment'].posts,bins=[0,1,4,9,np.inf],labels=['1','2–4','5–9','10+'])
label_support=counts['sentiment'].assign(support_band=support_bands).groupby('support_band',observed=False).agg(
    labels=('sentiment','size'),posts=('posts','sum')).reset_index()
save_table('label_support_bands',label_support)
diagnostics=pd.DataFrame([
    {'measure':'distinct_users','value':df.user.nunique()},
    {'measure':'distinct_text_groups','value':df.text_group_id.nunique()},
    {'measure':'repeated_text_rows','value':int(df.repeated_text.sum())},
    {'measure':'repeated_text_groups','value':df.loc[df.repeated_text,'text_group_id'].nunique()},
    {'measure':'distinct_hashtags','value':tag_frequency.hashtag.nunique()},
    {'measure':'post_hashtag_rows','value':len(tags)},
    {'measure':'url_flagged_posts','value':int(df.has_url.sum())},
    {'measure':'mention_flagged_posts','value':int(df.has_mention.sum())},
    {'measure':'posts_with_listed_negation_cues','value':int(df.negation_cue_count.gt(0).sum())},
])
save_table('text_and_identifier_diagnostics',diagnostics)
display(diagnostics)
display(coverage)
display(label_support)

,measure,value
0,distinct_users,670
1,distinct_text_groups,706
2,repeated_text_rows,8
3,repeated_text_groups,4
4,distinct_hashtags,975
5,post_hashtag_rows,1420
6,url_flagged_posts,0
7,mention_flagged_posts,0
8,posts_with_listed_negation_cues,19


,year,posts,first_timestamp,last_timestamp
0,2010,3,2010-05-15 15:30:00,2010-11-12 20:20:00
1,2011,4,2011-06-20 14:45:00,2011-09-22 19:30:00
2,2012,4,2012-02-18 14:30:00,2012-11-25 10:00:00
3,2013,4,2013-04-05 19:30:00,2013-12-05 16:00:00
4,2014,4,2014-06-03 08:20:00,2014-09-14 08:30:00
5,2015,19,2015-06-28 20:00:00,2015-12-22 14:20:00
6,2016,37,2016-02-10 19:45:00,2016-12-25 19:30:00
7,2017,42,2017-01-12 12:00:00,2017-12-28 19:15:00
8,2018,54,2018-02-05 15:15:00,2018-12-15 20:15:00
9,2019,70,2019-01-10 14:45:00,2019-12-28 14:30:00


,support_band,labels,posts
0,1,81,81
1,2–4,69,190
2,5–9,30,193
3,10+,11,246


## 6 Documentation export and final checks
Keep the cleaned input unchanged. Export an exploration copy with six clearly
documented added fields. Derived fields are candidates for inspection, not a final
feature set. The main classification target and any polarity mapping remain future
team decisions. No new target values are created here.

In [7]:
original_dictionary=pd.read_csv(BASE/'data/input_data_dictionary.csv')
definitions=original_dictionary.set_index('field').definition.to_dict()
definitions.update({
    'hour_period':'Source hour intervals: 00–05, 06–11, 12–17 or 18–23. Timezone unknown.',
    'is_weekend':'True when timestamp weekday is Saturday or Sunday. NumPy membership operation.',
    'log1p_interactions':'Natural log of 1 + likes + retweets. NumPy numeric transformation for exploration only.',
    'has_url':'True for text matching https?:// or www. (case insensitive). Limited lexical pattern.',
    'has_mention':'True for an at-sign followed by word characters outside a word. Limited lexical pattern.',
    'negation_cue_count':"Count of no, not, never, neither, nor, cannot or n’t contractions. Limited lexical pattern.",
})
data_dictionary=pd.DataFrame([{'feature':c,'role':'Source or previously derived' if c in source else 'New exploration feature',
    'dtype_in_memory':str(df[c].dtype),'definition':definitions[c]} for c in df])
save_table('exploration_data_dictionary',data_dictionary)
exported=df.copy()
exported['hashtag_list']=exported.hashtag_list.map(lambda x:json.dumps(x,ensure_ascii=False))
exported.to_csv(BASE/'data/posts_feature_exploration.csv',index=False,date_format='%Y-%m-%d %H:%M:%S')
loaded=pd.read_csv(BASE/'data/posts_feature_exploration.csv')
assert loaded.shape==(710,32)
assert loaded.source_record.tolist()==source.source_record.tolist()
pd.testing.assert_frame_equal(loaded[source.columns],source,check_dtype=False)
assert df.hashtag_count.eq(df.hashtag_list.str.len()).all()
assert len(tags)==int(df.hashtag_count.sum())
assert not df.groupby('text_group_id',observed=True).sentiment.nunique().gt(1).any()
assert hashlib.sha256(INPUT.read_bytes()).hexdigest()==input_hash
results={
    'source_rows':len(source),'source_columns':len(source.columns),'exploration_rows':len(df),'exploration_columns':len(df.columns),
    'added_columns':derived_cols,'missing_input_cells':int(source.isna().sum().sum()),
    'sentiment_labels':df.sentiment.nunique(),'singleton_labels':int(counts['sentiment'].posts.eq(1).sum()),
    'platforms':df.platform.nunique(),'location_labels':df.country.nunique(),'users':df.user.nunique(),
    'distinct_hashtags':len(tag_frequency),'post_hashtag_rows':len(tags),
    'url_flagged_posts':int(df.has_url.sum()),'mention_flagged_posts':int(df.has_mention.sum()),
    'posts_with_listed_negation_cues':int(df.negation_cue_count.gt(0).sum()),
    'constant_features':constants.feature.tolist(),'input_sha256':input_hash,
    'date_min':str(df.timestamp.min()),'date_max':str(df.timestamp.max()),
    'validation':'All reconciliation and reload checks passed',
    'charts_created':0,'models_trained':0,'hypothesis_tests_run':0,
    'versions':{'python':__import__('sys').version.split()[0],'pandas':pd.__version__,'numpy':np.__version__},
}
(BASE/'validation_results.json').write_text(json.dumps(results,indent=2))
print(json.dumps(results,indent=2))

{
  "source_rows": 710,
  "source_columns": 26,
  "exploration_rows": 710,
  "exploration_columns": 32,
  "added_columns": [
    "hour_period",
    "is_weekend",
    "log1p_interactions",
    "has_url",
    "has_mention",
    "negation_cue_count"
  ],
  "missing_input_cells": 0,
  "sentiment_labels": 191,
  "singleton_labels": 81,
  "platforms": 3,
  "location_labels": 33,
  "users": 670,
  "distinct_hashtags": 975,
  "post_hashtag_rows": 1420,
  "url_flagged_posts": 0,
  "mention_flagged_posts": 0,
  "posts_with_listed_negation_cues": 19,
  "constant_features": [
    "hashtag_count",
    "has_url",
    "has_mention"
  ],
  "input_sha256": "4c38dc7cc2a8a06a96c13e989541c024ce31e9ee6af890a5f429fbe1f892ec6a",
  "date_min": "2010-05-15 15:30:00",
  "date_max": "2023-10-22 20:45:00",
  "validation": "All reconciliation and reload checks passed",
  "charts_created": 0,
  "models_trained": 0,
  "hypothesis_tests_run": 0,
  "versions": {
    "python": "3.12.14",
    "pandas": "2.2.3",
    "num

## Handoff to the EDA and visualization team
The tables are inputs for colleagues to choose charts and research questions.
Every grouped table retains denominators or record counts. Confirm how to handle
small sentiment classes and geographic levels before comparisons. Simple aggregates
alone do not establish differences, trends or causal effects in a real population.

Keep original supplied sentiment values separate from later reviewed polarity labels.
Preserve expressive text and use text_group_id when designing future data splits.
Any scaling, vocabulary fitting, encoding or resampling for modeling must be learned
within the training split in later work. The generated features have not been selected
or validated for predictive performance.